# NatGasMoE Classification + Positioning — Optuna Optimization

Regime-Aware Heterogeneous MoE (TCN + MDN) with:
- **4-class quartile classification** (<25th / <50th / >50th / >75th percentile return)
- **tanh positioning head** mapping class probabilities to exposure in [-1, +1]
- **Weather features** (HDD/CDD degree days + spline HDD basis)

Pipeline:
1. Load NG futures OHLCV + EIA storage + weather
2. Build datasets with `NGMoEDataBuilder` (technicals + weather + regime + 4-class target)
3. Optuna search over MoE architecture + positioning loss weights
4. Train final model, diagnostics, backtest

In [ ]:
import os
import sys
import copy
import math
import time as _time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader

try:
    import optuna
    from optuna.pruners import MedianPruner
    from optuna.samplers import TPESampler
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    _OPTUNA = True
except ImportError:
    print('optuna not installed -- pip install optuna')
    _OPTUNA = False

warnings.filterwarnings('ignore')

print(f'PyTorch : {torch.__version__}')
print(f'Optuna  : {optuna.__version__ if _OPTUNA else "N/A"}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

In [ ]:
from CTAFlow.models.deep_learning.multi_branch.ng_moe import (
    MoEConfig,
    NatGasMoE,
    MoELoss,
)
from CTAFlow.models.deep_learning.multi_branch.ng_moe_dataset import (
    NGMoEDataConfig,
    NGMoEDataBuilder,
    NGMoEWindowDataset,
    REGIME_COLS,
    build_datasets,
)

# macrOS-Int for EIA + weather
sys.path.insert(0, r'C:\Users\nicho\PycharmProjects\macrOS-Int')
from MacrOSINT.data.sources.eia.api_tools import NatGasHelper
from MacrOSINT.models.energy.natgas_storage_forecast import (
    NatGasStorageForecaster,
    fetch_storage_data,
    ConsensusForecast,
)

# Device
def _select_device():
    if not torch.cuda.is_available():
        return 'cpu'
    try:
        t = torch.zeros(1, device='cuda')
        _ = t + 1
        return 'cuda'
    except RuntimeError as e:
        print(f'CUDA unusable ({e}) -- CPU fallback')
        return 'cpu'

DEVICE = _select_device()
print(f'Device: {DEVICE}')

## 1. Configuration

In [ ]:
# --- Paths (adjust for Colab / RunPod / local) ---
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    SAVE_DIR    = Path('/content/drive/MyDrive/results/ng_moe_cls')
    EIA_HDF     = '/content/drive/MyDrive/model_data/ng_eia_cache.hdf'
    WEATHER_HDF = '/content/drive/MyDrive/model_data/weather.hdf'
else:
    SAVE_DIR    = Path('/workspace/results/ng_moe_cls')
    EIA_HDF     = '/workspace/model_data/ng_eia_cache.hdf'
    WEATHER_HDF = '/workspace/model_data/weather.hdf'

SAVE_DIR.mkdir(parents=True, exist_ok=True)

# --- Classification config ---
N_CLASSES = 4           # quartile-based: <25 / <50 / >50 / >75
USE_POSITIONING = True  # tanh exposure head
TARGET_HORIZON = 1      # 1-day forward return

# --- Optuna ---
N_TRIALS       = 40
MAX_EPOCHS_OPT = 80
OPT_PATIENCE   = 10

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Save dir    : {SAVE_DIR}')
print(f'EIA HDF     : {EIA_HDF}')
print(f'Weather HDF : {WEATHER_HDF}')
print(f'Classes     : {N_CLASSES}')
print(f'Positioning : {USE_POSITIONING}')

## 2. Load Price Data

In [ ]:
import yfinance as yf

price_df = yf.download('NG=F', start='2010-01-01')
if isinstance(price_df.columns, pd.MultiIndex):
    price_df.columns = price_df.columns.get_level_values(0)
price_df.columns = [c.lower() for c in price_df.columns]
price_df = price_df.dropna(subset=['close'])

print(f'Price data : {price_df.shape}')
print(f'Date range : {price_df.index[0].date()} to {price_df.index[-1].date()}')
price_df.tail(3)

## 3. Load EIA Storage + Consensus

In [ ]:
START = price_df.index[0].strftime('%Y-%m')
END   = price_df.index[-1].strftime('%Y-%m')

eia_cache    = NatGasStorageForecaster.load_eia_cache(hdf_path=EIA_HDF)
storage_wkly = eia_cache.get('storage')

if storage_wkly is not None and not storage_wkly.empty:
    print(f'EIA storage from cache : {storage_wkly.shape}')
else:
    ng_helper    = NatGasHelper()
    storage_wkly = fetch_storage_data(ng_helper, start=START, end=END)
    NatGasStorageForecaster.save_eia_cache(storage=storage_wkly, hdf_path=EIA_HDF)
    print(f'EIA storage from API   : {storage_wkly.shape}')

# Consensus forecast for surprise features
try:
    cf = ConsensusForecast()
    cf.fit(storage_wkly['storage_change'])
    surprise_df = cf.transform()
    storage_wkly = storage_wkly.join(
        surprise_df[['consensus_est', 'surprise']], how='left'
    )
    print(f'ConsensusForecast added')
except Exception as e:
    print(f'ConsensusForecast failed ({e}) -- using rolling 4-week proxy')
    chg = storage_wkly['storage_change']
    storage_wkly['consensus_est'] = chg.rolling(4).mean()
    storage_wkly['surprise']      = chg - storage_wkly['consensus_est']

print(f'Weekly columns: {storage_wkly.columns.tolist()}')
storage_wkly.tail(3)

## 4. Load Population-Weighted Weather

HDD/CDD degree-day features with spline HDD basis for non-linear cold demand response.

In [ ]:
daily_weather = NatGasStorageForecaster.load_weather_hdf(hdf_path=WEATHER_HDF)

if daily_weather is not None and not daily_weather.empty:
    daily_weather = daily_weather[
        (daily_weather.index >= price_df.index[0]) &
        (daily_weather.index <= price_df.index[-1])
    ]
    print(f'Weather from cache : {daily_weather.shape}')
    print(f'Date range : {daily_weather.index[0].date()} to {daily_weather.index[-1].date()}')
    print(f'Columns    : {daily_weather.columns.tolist()}')
else:
    from MacrOSINT.models.weather.population_weather import PopulationWeatherGrid
    print('No weather cache -- fetching via PopulationWeatherGrid...')
    forecaster = NatGasStorageForecaster()
    forecaster.setup()
    daily_weather = forecaster._fetch_weather_by_epoch(
        price_df.index[0].date(), price_df.index[-1].date()
    )
    NatGasStorageForecaster.save_weather_hdf(daily_weather, hdf_path=WEATHER_HDF)
    print(f'Weather fetched and cached: {daily_weather.shape}')

daily_weather.tail(3)

## 5. Build Datasets

4-class quartile classification on expanding quantile boundaries (causal, no lookahead).
Weather features (HDD, CDD, spline basis, momentum) are included via `daily_weather`.

In [ ]:
data_cfg = NGMoEDataConfig(
    seq_len=20,
    ae_window=21,
    target='price_return',
    target_horizon=TARGET_HORIZON,
    n_classes=N_CLASSES,
)

train_ds, val_ds, test_ds, meta = build_datasets(
    price_df, storage_wkly,
    daily_weather=daily_weather,
    config=data_cfg,
    train_frac=0.70,
    val_frac=0.15,
    monday_only=False,
)

n_features = meta['n_features']
print(f'Target       : {data_cfg.target} (horizon={data_cfg.target_horizon}d)')
print(f'Classes      : {N_CLASSES} (quartile distribution)')
print(f'Features     : {n_features}')
print(f'Regime cols  : {len(meta["regime_cols"])}')
print(f'Train samples: {len(train_ds)}')
print(f'Val samples  : {len(val_ds)}')
print(f'Test samples : {len(test_ds)}')

for split_name, (start, end, count) in meta['splits'].items():
    print(f'  {split_name:5s}: {start.date()} - {end.date()}  ({count} days)')

# Verify classification target
sample = train_ds[0]
print(f'\nSample length: {len(sample)} elements')
if len(sample) == 5:
    x_seq, ae_input, y_ret, y_std, y_class = sample
    print(f'x_seq={tuple(x_seq.shape)}, ae_input={tuple(ae_input.shape)}, '
          f'y_ret={y_ret.item():.4f}, y_std={y_std.item():.4f}, y_class={y_class.item()}')
else:
    x_seq, ae_input, y_ret, y_std = sample
    print(f'x_seq={tuple(x_seq.shape)}, ae_input={tuple(ae_input.shape)}')

# Weather feature check
weather_cols = [c for c in meta['feature_cols'] if c.startswith('dd_') or c.startswith('wtd_') or 'hdd' in c.lower()]
print(f'\nWeather features ({len(weather_cols)}): {weather_cols}')

# Class distribution
if train_ds.has_classes:
    train_classes = train_ds.y_class[train_ds.indices]
    for c in range(N_CLASSES):
        pct = (train_classes == c).mean() * 100
        print(f'  Class {c}: {pct:.1f}%')

## 6. Selection Score

Composite metric balancing classification accuracy, Sharpe, and positioning PnL.

In [ ]:
def compute_val_metrics(model, val_loader, loss_fn, device):
    """Evaluate model on validation set, return metrics dict."""
    model.eval()
    all_losses = []
    all_positions = []
    all_returns = []
    all_correct = []
    n_total = 0

    with torch.no_grad():
        for batch in val_loader:
            if len(batch) == 5:
                x_seq, ae_in, y_ret, y_std, y_cls = batch
                y_cls = y_cls.to(device)
            else:
                x_seq, ae_in, y_ret, y_std = batch
                y_cls = None

            x_seq = x_seq.to(device)
            ae_in = ae_in.to(device)
            y_ret = y_ret.to(device)
            y_std = y_std.to(device)

            out = model(x_seq, ae_in)
            losses = loss_fn(out, y_ret, y_std, y_cls)
            all_losses.append(losses['total_loss'].item() * len(y_ret))
            n_total += len(y_ret)

            if 'position' in out:
                pos = out['position'].cpu().numpy()
                ret = y_ret.cpu().numpy()
                all_positions.append(pos)
                all_returns.append(ret)

            if 'class_logits' in out and y_cls is not None:
                preds = out['class_logits'].argmax(dim=-1)
                all_correct.append((preds == y_cls).float().cpu().numpy())

    metrics = {'val_loss': sum(all_losses) / max(n_total, 1)}

    if all_correct:
        metrics['accuracy'] = np.concatenate(all_correct).mean()

    if all_positions:
        positions = np.concatenate(all_positions)
        returns = np.concatenate(all_returns)
        strategy_ret = positions * returns
        metrics['mean_strategy_ret'] = strategy_ret.mean()
        std = strategy_ret.std()
        metrics['sharpe'] = (strategy_ret.mean() / std * np.sqrt(252)) if std > 0 else 0.0
        # Sortino
        downside = strategy_ret[strategy_ret < 0]
        down_std = downside.std() if len(downside) > 1 else 1e-6
        metrics['sortino'] = (strategy_ret.mean() / down_std * np.sqrt(252)) if down_std > 0 else 0.0
        # Profit factor
        gains = strategy_ret[strategy_ret > 0].sum()
        losses_sum = abs(strategy_ret[strategy_ret < 0].sum())
        metrics['profit_factor'] = gains / max(losses_sum, 1e-8)
        metrics['mean_abs_position'] = np.abs(positions).mean()

    return metrics


def selection_score(metrics: dict) -> float:
    """Composite score for Optuna maximization."""
    if USE_POSITIONING:
        sharpe = metrics.get('sharpe', 0.0)
        sortino = metrics.get('sortino', 0.0)
        pf = metrics.get('profit_factor', 1e-8)
        acc = metrics.get('accuracy', 0.25)
        return (
            0.25 * sharpe
            + 0.30 * sortino
            + 0.20 * math.log(max(pf, 1e-8))
            + 0.25 * (acc - 0.25) * 10  # above chance
        )
    else:
        # Classification only: accuracy + return correlation
        acc = metrics.get('accuracy', 0.25)
        return -metrics['val_loss'] + 5.0 * (acc - 0.25)

print('Selection score defined')

## 7. Optuna Objective

In [ ]:
_trial_log = []


def _make_loaders(train_ds, val_ds, bs):
    tr = DataLoader(train_ds, batch_size=bs, shuffle=True, drop_last=True)
    va = DataLoader(val_ds, batch_size=bs, shuffle=False, drop_last=False)
    return tr, va


def objective(trial):
    t0 = _time.time()

    # --- Architecture ---
    d_latent       = trial.suggest_categorical('d_latent', [16, 32, 64])
    d_ae_hidden    = trial.suggest_categorical('d_ae_hidden', [64, 128, 256])
    n_tcn_experts  = trial.suggest_int('n_tcn_experts', 2, 5)
    n_mdn_experts  = trial.suggest_int('n_mdn_experts', 1, 4)
    top_k          = trial.suggest_int('top_k', 2, min(n_tcn_experts + n_mdn_experts, 5))
    tcn_width      = trial.suggest_categorical('tcn_width', [32, 64, 128])
    tcn_depth      = trial.suggest_int('tcn_depth', 2, 4)
    mdn_hidden     = trial.suggest_categorical('mdn_hidden', [32, 64, 128])
    mdn_n_comp     = trial.suggest_int('mdn_n_components', 2, 6)
    shared_dim     = trial.suggest_categorical('shared_expert_dim', [32, 64, 128])
    dropout        = trial.suggest_float('dropout', 0.05, 0.4)
    pos_hidden     = trial.suggest_categorical('positioning_hidden_dim', [16, 32, 64])

    # --- Loss weights ---
    ce_weight      = trial.suggest_float('ce_weight', 0.3, 3.0, log=True)
    pnl_weight     = trial.suggest_float('positioning_pnl_weight', 0.1, 2.0, log=True)
    nll_weight     = trial.suggest_float('nll_weight', 0.01, 0.5, log=True)
    kl_weight      = trial.suggest_float('kl_weight', 0.001, 0.1, log=True)
    recon_weight   = trial.suggest_float('recon_weight', 0.01, 0.5, log=True)
    balance_w      = trial.suggest_float('load_balance_weight', 0.001, 0.1, log=True)
    entropy_w      = trial.suggest_float('entropy_reg_weight', 0.001, 0.1, log=True)
    tc_cost        = trial.suggest_float('tc_cost', 0.0, 0.001)

    # --- Training ---
    lr             = trial.suggest_float('lr', 1e-4, 3e-3, log=True)
    weight_decay   = trial.suggest_float('weight_decay', 1e-5, 1e-2, log=True)
    batch_size     = trial.suggest_categorical('batch_size', [16, 32, 64])

    config = MoEConfig(
        n_features=n_features,
        seq_len=data_cfg.seq_len,
        f_ae=12,
        ae_window=data_cfg.ae_window,
        d_latent=d_latent,
        d_ae_hidden=d_ae_hidden,
        kl_weight=kl_weight,
        recon_weight=recon_weight,
        n_tcn_experts=n_tcn_experts,
        n_mdn_experts=n_mdn_experts,
        top_k=top_k,
        shared_expert_dim=shared_dim,
        tcn_channels=[tcn_width] * tcn_depth,
        mdn_hidden_dims=[mdn_hidden, mdn_hidden // 2],
        mdn_n_components=mdn_n_comp,
        dropout=dropout,
        load_balance_weight=balance_w,
        entropy_reg_weight=entropy_w,
        nll_weight=nll_weight,
        n_classes=N_CLASSES,
        use_positioning_head=USE_POSITIONING,
        positioning_hidden_dim=pos_hidden,
        ce_weight=ce_weight,
        positioning_pnl_weight=pnl_weight,
        tc_cost=tc_cost,
    )

    model = NatGasMoE(config).to(DEVICE)
    loss_fn = MoELoss(config)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)

    train_loader, val_loader = _make_loaders(train_ds, val_ds, batch_size)

    best_score = float('-inf')
    patience_cnt = 0

    for epoch in range(1, MAX_EPOCHS_OPT + 1):
        # --- Train ---
        model.train()
        for batch in train_loader:
            if len(batch) == 5:
                x_seq, ae_in, y_ret, y_std, y_cls = batch
                y_cls = y_cls.to(DEVICE)
            else:
                x_seq, ae_in, y_ret, y_std = batch
                y_cls = None

            x_seq = x_seq.to(DEVICE)
            ae_in = ae_in.to(DEVICE)
            y_ret = y_ret.to(DEVICE)
            y_std = y_std.to(DEVICE)

            optimizer.zero_grad()
            out = model(x_seq, ae_in)
            losses = loss_fn(out, y_ret, y_std, y_cls)
            losses['total_loss'].backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        # --- Validate ---
        metrics = compute_val_metrics(model, val_loader, loss_fn, DEVICE)
        score = selection_score(metrics)

        trial.report(score, epoch)
        if trial.should_prune():
            _trial_log.append({
                'trial': trial.number, 'status': 'PRUNED',
                'epoch': epoch, 'score': score,
                'val_loss': metrics['val_loss'],
                'time': _time.time() - t0,
            })
            raise optuna.TrialPruned()

        if score > best_score:
            best_score = score
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= OPT_PATIENCE:
                break

    _trial_log.append({
        'trial': trial.number, 'status': 'COMPLETE',
        'epoch': epoch, 'score': best_score,
        'val_loss': metrics['val_loss'],
        'accuracy': metrics.get('accuracy', 0),
        'sharpe': metrics.get('sharpe', 0),
        'sortino': metrics.get('sortino', 0),
        'profit_factor': metrics.get('profit_factor', 0),
        'time': _time.time() - t0,
    })
    return best_score

print(f'Objective defined (n_features={n_features})')

## 8. Run Optuna Optimization

In [ ]:
pos_tag = '_pos' if USE_POSITIONING else ''
STUDY_NAME = f'ng_moe_cls{N_CLASSES}{pos_tag}'

study = optuna.create_study(
    study_name=STUDY_NAME,
    direction='maximize',
    sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(n_startup_trials=3, n_warmup_steps=5),
)

print(f'Study   : {STUDY_NAME}')
print(f'Trials  : {N_TRIALS}')
print(f'Classes : {N_CLASSES}')
print(f'Head    : {"tanh positioning" if USE_POSITIONING else "classification only"}')
print('-' * 60)

In [ ]:
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nBest trial: #{study.best_trial.number}')
print(f'Best score: {study.best_value:.4f}')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
# Save study artifacts
import json

best_params = dict(study.best_params)
best_params['best_value'] = study.best_value
best_params['n_classes'] = N_CLASSES
best_params['use_positioning'] = USE_POSITIONING
best_params['n_features'] = n_features
best_params['target_horizon'] = TARGET_HORIZON

with open(SAVE_DIR / 'best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)

trial_df = pd.DataFrame(_trial_log)
trial_df.to_csv(SAVE_DIR / 'trial_log.csv', index=False)
print(f'Saved to {SAVE_DIR}')
trial_df.sort_values('score', ascending=False).head(10)

## 9. Optuna Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Optimization history
comp = trial_df[trial_df['status'] == 'COMPLETE'].copy()
best_so_far = comp['score'].expanding().max()
axes[0, 0].scatter(comp['trial'], comp['score'], s=20, alpha=0.6, label='trial score')
axes[0, 0].plot(comp['trial'].values, best_so_far.values, color='red', lw=1.5, label='best so far')
axes[0, 0].set_xlabel('Trial'); axes[0, 0].set_ylabel('Score')
axes[0, 0].set_title('Optimization History'); axes[0, 0].legend(fontsize=8)

# 2. Param importances
try:
    imp = optuna.importance.get_param_importances(study)
    names = list(imp.keys())[:12]
    vals  = list(imp.values())[:12]
    axes[0, 1].barh(names, vals, color='steelblue', edgecolor='black')
    axes[0, 1].set_xlabel('Importance')
    axes[0, 1].set_title('Top Hyperparameter Importance')
except Exception:
    axes[0, 1].text(0.5, 0.5, 'Not enough data', ha='center', va='center',
                    transform=axes[0, 1].transAxes)

# 3. Score components (accuracy, sharpe, sortino)
if 'accuracy' in comp.columns:
    axes[1, 0].scatter(comp['accuracy'], comp['score'], s=20, alpha=0.6, c='green')
    axes[1, 0].set_xlabel('Val Accuracy'); axes[1, 0].set_ylabel('Score')
    axes[1, 0].set_title('Score vs Accuracy')

if 'sharpe' in comp.columns:
    axes[1, 1].scatter(comp['sharpe'], comp['sortino'], s=20, alpha=0.6,
                       c=comp['score'], cmap='viridis')
    axes[1, 1].set_xlabel('Sharpe'); axes[1, 1].set_ylabel('Sortino')
    axes[1, 1].set_title('Sharpe vs Sortino (color=score)')
    plt.colorbar(axes[1, 1].collections[0], ax=axes[1, 1], label='Score')

plt.tight_layout()
plt.savefig(SAVE_DIR / 'optuna_viz.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Train Final Model with Best Parameters

In [ ]:
bp = study.best_params
print('Best parameters:')
for k, v in sorted(bp.items()):
    print(f'  {k}: {v}')

final_cfg = MoEConfig(
    n_features=n_features,
    seq_len=data_cfg.seq_len,
    f_ae=12,
    ae_window=data_cfg.ae_window,
    d_latent=bp['d_latent'],
    d_ae_hidden=bp['d_ae_hidden'],
    kl_weight=bp['kl_weight'],
    recon_weight=bp['recon_weight'],
    n_tcn_experts=bp['n_tcn_experts'],
    n_mdn_experts=bp['n_mdn_experts'],
    top_k=bp['top_k'],
    shared_expert_dim=bp['shared_expert_dim'],
    tcn_channels=[bp['tcn_width']] * bp['tcn_depth'],
    mdn_hidden_dims=[bp['mdn_hidden'], bp['mdn_hidden'] // 2],
    mdn_n_components=bp['mdn_n_components'],
    dropout=bp['dropout'],
    load_balance_weight=bp['load_balance_weight'],
    entropy_reg_weight=bp['entropy_reg_weight'],
    nll_weight=bp['nll_weight'],
    n_classes=N_CLASSES,
    use_positioning_head=USE_POSITIONING,
    positioning_hidden_dim=bp['positioning_hidden_dim'],
    ce_weight=bp['ce_weight'],
    positioning_pnl_weight=bp['positioning_pnl_weight'],
    tc_cost=bp['tc_cost'],
)

model = NatGasMoE(final_cfg).to(DEVICE)
loss_fn = MoELoss(final_cfg)
optimizer = optim.AdamW(model.parameters(), lr=bp['lr'], weight_decay=bp['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=50, T_mult=2)

FINAL_BS = bp['batch_size']
train_loader = DataLoader(train_ds, batch_size=FINAL_BS, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds, batch_size=FINAL_BS, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds, batch_size=FINAL_BS, shuffle=False, drop_last=False)

print(f'\nModel params : {sum(p.numel() for p in model.parameters()):,}')
print(f'Train batches: {len(train_loader)}, Val: {len(val_loader)}, Test: {len(test_loader)}')

In [ ]:
MAX_EPOCHS = 200
PATIENCE   = 20

loss_keys = ['total_loss', 'return_loss', 'vol_loss', 'mdn_nll_loss',
             'balance_loss', 'entropy_loss', 'ae_recon_loss', 'ae_kl_loss']
if N_CLASSES > 0:
    loss_keys.extend(['ce_loss', 'class_accuracy'])
if USE_POSITIONING:
    loss_keys.extend(['positioning_loss', 'mean_position', 'mean_strategy_ret'])

history = {f'train_{k}': [] for k in loss_keys}
history.update({f'val_{k}': [] for k in loss_keys})
history['shared_gate'] = []

best_val_loss = float('inf')
best_state    = None
patience_cnt  = 0

for epoch in range(1, MAX_EPOCHS + 1):
    # --- Train ---
    model.train()
    epoch_train = {k: [] for k in loss_keys}
    for batch in train_loader:
        if len(batch) == 5:
            x_seq, ae_in, y_ret, y_std, y_cls = batch
            y_cls = y_cls.to(DEVICE)
        else:
            x_seq, ae_in, y_ret, y_std = batch
            y_cls = None

        x_seq = x_seq.to(DEVICE)
        ae_in = ae_in.to(DEVICE)
        y_ret = y_ret.to(DEVICE)
        y_std = y_std.to(DEVICE)

        optimizer.zero_grad()
        out = model(x_seq, ae_in)
        losses = loss_fn(out, y_ret, y_std, y_cls)
        losses['total_loss'].backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        for k in loss_keys:
            if k in losses:
                v = losses[k]
                epoch_train[k].append(v.item() if torch.is_tensor(v) else v)
    scheduler.step()

    # --- Validate ---
    model.eval()
    epoch_val = {k: [] for k in loss_keys}
    gate_vals = []
    with torch.no_grad():
        for batch in val_loader:
            if len(batch) == 5:
                x_seq, ae_in, y_ret, y_std, y_cls = batch
                y_cls = y_cls.to(DEVICE)
            else:
                x_seq, ae_in, y_ret, y_std = batch
                y_cls = None

            x_seq = x_seq.to(DEVICE)
            ae_in = ae_in.to(DEVICE)
            y_ret = y_ret.to(DEVICE)
            y_std = y_std.to(DEVICE)

            out = model(x_seq, ae_in)
            losses = loss_fn(out, y_ret, y_std, y_cls)
            for k in loss_keys:
                if k in losses:
                    v = losses[k]
                    epoch_val[k].append(v.item() if torch.is_tensor(v) else v)
            gate_vals.append(out['shared_gate'].item())

    # Log
    for k in loss_keys:
        history[f'train_{k}'].append(np.mean(epoch_train[k]) if epoch_train[k] else 0)
        history[f'val_{k}'].append(np.mean(epoch_val[k]) if epoch_val[k] else 0)
    history['shared_gate'].append(np.mean(gate_vals))

    val_loss = history['val_total_loss'][-1]

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_state = copy.deepcopy(model.state_dict())
        patience_cnt = 0
    else:
        patience_cnt += 1

    if epoch % 10 == 0 or patience_cnt == 0:
        extras = ''
        if 'val_class_accuracy' in history:
            extras += f'  acc={history["val_class_accuracy"][-1]:.3f}'
        if 'val_mean_strategy_ret' in history and history['val_mean_strategy_ret'][-1] != 0:
            extras += f'  strat_ret={history["val_mean_strategy_ret"][-1]:.5f}'
        print(f'Epoch {epoch:3d}  train={history["train_total_loss"][-1]:.5f}  '
              f'val={val_loss:.5f}{extras}  gate={history["shared_gate"][-1]:.3f}'
              f'{"  *" if patience_cnt == 0 else ""}')

    if patience_cnt >= PATIENCE:
        print(f'Early stopping at epoch {epoch}')
        break

print(f'\nBest val loss: {best_val_loss:.5f}')

In [ ]:
# Load best state
if best_state:
    model.load_state_dict(best_state)
    print('Loaded best model state')

## 11. Training Diagnostics

In [ ]:
ep = range(1, len(history['train_total_loss']) + 1)
n_plots = 4 if USE_POSITIONING else 3
fig, axes = plt.subplots(2, n_plots, figsize=(5 * n_plots, 8))

# Row 1: Core losses
axes[0, 0].plot(ep, history['train_total_loss'], label='Train')
axes[0, 0].plot(ep, history['val_total_loss'], label='Val')
axes[0, 0].set_title('Total Loss'); axes[0, 0].legend()

axes[0, 1].plot(ep, history['train_return_loss'], label='Train')
axes[0, 1].plot(ep, history['val_return_loss'], label='Val')
axes[0, 1].set_title('Return Loss (Huber)'); axes[0, 1].legend()

axes[0, 2].plot(ep, history['train_vol_loss'], label='Train')
axes[0, 2].plot(ep, history['val_vol_loss'], label='Val')
axes[0, 2].set_title('Vol Loss'); axes[0, 2].legend()

if USE_POSITIONING:
    axes[0, 3].plot(ep, history['val_positioning_loss'], color='purple')
    axes[0, 3].set_title('Positioning Loss (Val)')

# Row 2: Classification + routing
if N_CLASSES > 0 and 'val_class_accuracy' in history:
    axes[1, 0].plot(ep, history['train_class_accuracy'], label='Train')
    axes[1, 0].plot(ep, history['val_class_accuracy'], label='Val')
    axes[1, 0].axhline(1.0 / N_CLASSES, color='red', linestyle='--', label='Chance')
    axes[1, 0].set_title(f'{N_CLASSES}-Class Accuracy'); axes[1, 0].legend()
else:
    axes[1, 0].plot(ep, history['val_mdn_nll_loss'], color='purple')
    axes[1, 0].set_title('MDN NLL (Val)')

axes[1, 1].plot(ep, history['val_balance_loss'], label='Balance')
axes[1, 1].plot(ep, history['val_entropy_loss'], label='Entropy')
axes[1, 1].set_title('Router Regularization'); axes[1, 1].legend()

axes[1, 2].plot(ep, history['shared_gate'], color='steelblue')
axes[1, 2].set_title('Shared Gate'); axes[1, 2].set_ylabel('alpha')

if USE_POSITIONING and 'val_mean_strategy_ret' in history:
    axes[1, 3].plot(ep, np.cumsum(history['val_mean_strategy_ret']), color='green')
    axes[1, 3].set_title('Cumulative Val Strategy Return')
    axes[1, 3].axhline(0, color='black', linewidth=0.5)

plt.tight_layout()
plt.savefig(SAVE_DIR / 'training_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Router Utilization

In [ ]:
model.eval()
all_weights = []
all_z_regime = []

with torch.no_grad():
    for batch in val_loader:
        x_seq = batch[0].to(DEVICE)
        ae_in = batch[1].to(DEVICE)
        out = model(x_seq, ae_in)
        all_weights.append(out['router_weights'].cpu().numpy())
        all_z_regime.append(out['z_regime'].cpu().numpy())

weights_np = np.concatenate(all_weights)
z_regime_np = np.concatenate(all_z_regime)

n_tcn = final_cfg.n_tcn_experts
n_mdn = final_cfg.n_mdn_experts
expert_labels = [f'TCN_{i}' for i in range(n_tcn)] + [f'MDN_{i}' for i in range(n_mdn)]
avg_w = weights_np.mean(axis=0)

print(f'{"Expert":<8} {"Avg Weight":>10} {"Active %":>9}')
print('-' * 32)
for i, label in enumerate(expert_labels):
    active_pct = (weights_np[:, i] > 0).mean() * 100
    print(f'{label:<8} {avg_w[i]:>10.4f} {active_pct:>8.1f}%')

fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue'] * n_tcn + ['darkorange'] * n_mdn
ax.bar(expert_labels, avg_w, color=colors, edgecolor='black')
ax.axhline(1.0 / len(expert_labels), color='red', linestyle='--', label='uniform')
ax.set_ylabel('Average Router Weight')
ax.set_title('Expert Utilization (Val)')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 13. Prediction Analysis + Backtest

In [ ]:
def evaluate_split(ds, loader, name):
    """Run model on a dataset split, return results dict."""
    model.eval()
    preds_ret, preds_std, actuals_ret = [], [], []
    pred_classes, actual_classes = [], []
    positions = []

    with torch.no_grad():
        for batch in loader:
            if len(batch) == 5:
                x_seq, ae_in, y_ret, y_std, y_cls = batch
            else:
                x_seq, ae_in, y_ret, y_std = batch
                y_cls = None

            out = model(x_seq.to(DEVICE), ae_in.to(DEVICE))

            preds_ret.append(out['pred_return'].cpu().numpy())
            preds_std.append(out['pred_std'].cpu().numpy())
            actuals_ret.append(y_ret.numpy())

            if 'class_logits' in out:
                pred_classes.append(out['class_logits'].argmax(dim=-1).cpu().numpy())
            if y_cls is not None:
                actual_classes.append(y_cls.numpy())
            if 'position' in out:
                positions.append(out['position'].cpu().numpy())

    preds_ret = np.concatenate(preds_ret)
    actuals_ret = np.concatenate(actuals_ret)
    dates = [ds.get_date(i) for i in range(len(ds))]

    # Return prediction metrics
    mae = np.abs(preds_ret - actuals_ret).mean()
    corr = np.corrcoef(preds_ret, actuals_ret)[0, 1]
    dir_mask = np.abs(actuals_ret) > 1e-6
    dir_acc = (np.sign(preds_ret[dir_mask]) == np.sign(actuals_ret[dir_mask])).mean()

    print(f'\n=== {name} ({len(ds)} samples) ===')
    print(f'  MAE: {mae:.5f}  Corr: {corr:.4f}  Dir acc: {dir_acc:.3f}')

    # Classification metrics
    if pred_classes and actual_classes:
        pc = np.concatenate(pred_classes)
        ac = np.concatenate(actual_classes)
        cls_acc = (pc == ac).mean()
        print(f'  Class accuracy: {cls_acc:.3f} (chance={1/N_CLASSES:.3f})')
        # Per-class accuracy
        for c in range(N_CLASSES):
            mask = ac == c
            if mask.sum() > 0:
                print(f'    Class {c}: {(pc[mask] == c).mean():.3f} ({mask.sum()} samples)')

    # Positioning backtest
    if positions:
        pos = np.concatenate(positions)
        strat_ret = pos * actuals_ret
        cum_ret = np.cumsum(strat_ret)
        sharpe = strat_ret.mean() / strat_ret.std() * np.sqrt(252) if strat_ret.std() > 0 else 0
        downside = strat_ret[strat_ret < 0]
        sortino = strat_ret.mean() / downside.std() * np.sqrt(252) if len(downside) > 1 and downside.std() > 0 else 0
        max_dd = (cum_ret - np.maximum.accumulate(cum_ret)).min()
        print(f'  Positioning: Sharpe={sharpe:.3f}  Sortino={sortino:.3f}  MaxDD={max_dd:.4f}')
        print(f'  Mean |pos|={np.abs(pos).mean():.3f}  Total return={cum_ret[-1]:.4f}')
        return {'dates': dates, 'positions': pos, 'strategy_ret': strat_ret, 'cum_ret': cum_ret}

    return {'dates': dates, 'preds': preds_ret, 'actuals': actuals_ret}

val_results = evaluate_split(val_ds, val_loader, 'Validation')
test_results = evaluate_split(test_ds, test_loader, 'Test')

In [ ]:
# Backtest charts
if 'cum_ret' in test_results:
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))

    # Cumulative return
    axes[0, 0].plot(test_results['dates'], test_results['cum_ret'], color='green')
    axes[0, 0].axhline(0, color='black', linewidth=0.5)
    axes[0, 0].set_title('Test: Cumulative Strategy Return')
    axes[0, 0].set_ylabel('Cumulative log return')

    # Position distribution
    axes[0, 1].hist(test_results['positions'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    axes[0, 1].set_title('Test: Position Distribution')
    axes[0, 1].set_xlabel('Position [-1, +1]')

    # Rolling Sharpe (60-day)
    sr = pd.Series(test_results['strategy_ret'])
    roll_sharpe = sr.rolling(60).mean() / sr.rolling(60).std() * np.sqrt(252)
    axes[1, 0].plot(test_results['dates'], roll_sharpe.values, color='purple')
    axes[1, 0].axhline(0, color='black', linewidth=0.5)
    axes[1, 0].set_title('Test: Rolling 60d Sharpe')

    # Val cumulative return
    if 'cum_ret' in val_results:
        axes[1, 1].plot(val_results['dates'], val_results['cum_ret'], color='blue', label='Val')
        axes[1, 1].plot(test_results['dates'], test_results['cum_ret'], color='green', label='Test')
        axes[1, 1].axhline(0, color='black', linewidth=0.5)
        axes[1, 1].set_title('Val vs Test Cumulative Return')
        axes[1, 1].legend()

    plt.tight_layout()
    plt.savefig(SAVE_DIR / 'backtest_charts.png', dpi=150, bbox_inches='tight')
    plt.show()

## 14. Save Final Model

In [ ]:
import json
from dataclasses import asdict

model_path = SAVE_DIR / 'ng_moe_cls_best.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'config': asdict(final_cfg),
    'data_config': asdict(data_cfg),
    'best_params': best_params,
    'n_features': n_features,
    'feature_cols': meta['feature_cols'],
    'regime_cols': meta['regime_cols'],
}, model_path)

print(f'Model saved to {model_path}')
print(f'Config: {N_CLASSES} classes, positioning={USE_POSITIONING}')
print(f'Features: {n_features} technical + 12 regime')
weather_cols = [c for c in meta['feature_cols'] if c.startswith('dd_') or c.startswith('wtd_') or 'hdd' in c.lower()]
print(f'Weather features: {len(weather_cols)}')